## `nodeSelector` & node affinity

**`nodeSelector`** is the simplest pin — a map of labels that must *all* match a node:

```yaml
spec:
  nodeSelector: { disktype: ssd, gpu: "true" }
```

Only a node with both `disktype=ssd` *and* `gpu=true` qualifies. Node labels come from the admin (`kubectl label node worker-3 disktype=ssd`) or automatically from the kubelet (`kubernetes.io/hostname`, `.../os`, `.../arch`, `topology.kubernetes.io/zone|region`). Use `nodeSelector` when the rule is plain: "needs a node with X." (`spec.nodeName` is a bigger hammer — it *bypasses* the scheduler entirely; almost never in normal manifests.)

**Node affinity** is the grown-up version, in two flavours:

- **`requiredDuringSchedulingIgnoredDuringExecution`** — a hard requirement, but with the full set-based selector language.
- **`preferredDuringSchedulingIgnoredDuringExecution`** — a weighted soft preference; honoured if possible.

```yaml
affinity:
  nodeAffinity:
    requiredDuringSchedulingIgnoredDuringExecution:
      nodeSelectorTerms:
        - matchExpressions:
            - { key: topology.kubernetes.io/zone, operator: In, values: [us-east-1a, us-east-1b] }
            - { key: gpu, operator: Exists }
    preferredDuringSchedulingIgnoredDuringExecution:
      - weight: 50
        preference:
          matchExpressions: [{ key: instance-type, operator: In, values: [m5.xlarge] }]
```

*Must* be in zone 1a/1b **and** have a `gpu` label; *prefer* `m5.xlarge`. Operators: `In`, `NotIn`, `Exists`, `DoesNotExist`, `Gt`, `Lt`. **"IgnoredDuringExecution"** means affinity isn't re-checked once placed — a placed pod stays even if labels change.

Rule of thumb: single equality → `nodeSelector`; multi-condition/set-based → `nodeAffinity.required`; "try but don't refuse" → `preferred`. On our map both are the **nodeSelector** and **affinity** chips in the Scheduling box, feeding the scheduler's *filter* and *score* steps.